<a href="https://colab.research.google.com/github/kkumarisfdc/Kiran-s_Portfolio/blob/main/Find_PAM_and_sgRNA_sequence.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install biopython


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 32.9 MB/s eta 0:00:00


In [ ]:
from Bio import SeqIO
from Bio.Seq import Seq


In [ ]:
# Parsing Genbank files
from Bio import SeqIO, Entrez

# NCBI requires an email address for tracking automated API queries
Entrez.email = "your_email@example.com"

# Download GenBank record directly from NCBI (SARS-CoV-2 genome)
accession = "NC_045512.2"
print(f"Connecting to NCBI to fetch {accession}...")
handle = Entrez.efetch(db="nucleotide", id=accession, rettype="gb", retmode="text")

# Parse the GenBank record using SeqIO
genome_record = SeqIO.read(handle, "genbank")
handle.close()

# FIXED: Separated description text from the actual DNA sequence display
print(f"\nSuccessfully fetched {accession} from NCBI!")
print(f"Record ID:   {genome_record.id}")
print(f"Description: {genome_record.description}")
print(f"Total Length:{len(genome_record.seq):,} bp")
print(f"First 50 bp: {genome_record.seq[:50]}...")




Connecting to NCBI to fetch NC_045512.2...

Successfully fetched NC_045512.2 from NCBI!
Record ID:   NC_045512.2
Description: Severe acute respiratory syndrome coronavirus 2 isolate Wuhan-Hu-1, complete genome
Total Length:29,903 bp
First 50 bp: ATTAAAGGTTTATACCTTCCCAGGTAACAAACCAACCAACTTTCGATCTC...


In [ ]:
import re
import pandas as pd

# 1. Convert sequence to string
sequence_str = str(genome_record.seq).upper()

# 2. Extract targets
pam_pattern = r"(?=([A-Z]{20})(([A-Z])GG))"
matches = list(re.finditer(pam_pattern, sequence_str))

table_data = []

for index, match in enumerate(matches, start=1):
    spacer = match.group(1)
    full_pam = match.group(2)
    seed_region = spacer[-12:]

    # --- EVALUATE BIOLOGICAL EFFICIENCY ---

    # 1. Calculate Full Spacer GC Content Percentage
    gc_count = spacer.count('G') + spacer.count('C')
    gc_percent = (gc_count / len(spacer)) * 100

    # 2. Check for harmful homopolymer runs (TTTT or GGGG)
    has_poly_t = "TTTT" in spacer
    has_poly_g = "GGGG" in spacer
    has_poly_run = "Yes" if (has_poly_t or has_poly_g) else "No"

    # 3. Simple Scoring Metric (Ideal GC = 40-60% AND No Poly-Runs)
    is_ideal_gc = 40 <= gc_percent <= 60
    is_viable = is_ideal_gc and (has_poly_run == "No")

    # Assign a rank score (Higher is better)
    # Give priority to ideal GC content and penalize poly-runs
    score = 100 if is_viable else 30
    if not is_ideal_gc:
        score -= 40
    if has_poly_run == "Yes":
        score -= 50

    # Calculate coordinates
    pam_start = match.start() + 21
    pam_end = pam_start + 2

    table_data.append({
        "Target Site": index,
        "20 nt Target Sequence": spacer,
        "PAM": full_pam,
        "GC Content": f"{gc_percent:.1f}%",
        "Poly Run (T/G)": has_poly_run,
        "Efficiency Score": score,
        "Position": f"{pam_start}–{pam_end}"
    })

# Convert to DataFrame
df = pd.DataFrame(table_data)

# 4. FILTER & SORT: Show only viable candidates, sorted by highest efficiency score
best_targets = df[df["Efficiency Score"] >= 100].sort_values(by="Target Site").head(15)

# Display the clean filtered table with the default pandas index hidden
best_targets.style.hide(axis="index")


Target Site,20 nt Target Sequence,PAM,GC Content,Poly Run (T/G),Efficiency Score,Position
3,AAATCTGTGTGGCTGTCACT,CGG,45.0%,No,100,100–102
5,GTAACTCGTCTATCTTCTGC,AGG,45.0%,No,100,187–189
6,TATCTTCTGCAGGCTGCTTA,CGG,45.0%,No,100,197–199
7,AGCCGATCATCAGCACATCT,AGG,50.0%,No,100,235–237
8,CAGCACATCTAGGTTTCGTC,CGG,50.0%,No,100,245–247
9,AGCACATCTAGGTTTCGTCC,GGG,50.0%,No,100,246–248
10,TTCGTCCGGGTGTGACCGAA,AGG,60.0%,No,100,259–261
11,GGTGTGACCGAAAGGTAAGA,TGG,50.0%,No,100,267–269
12,AAGATGGAGAGCCTTGTCCC,TGG,55.0%,No,100,283–285
15,GACGTGCTCGTACGTGGCTT,TGG,60.0%,No,100,358–360


In [ ]:
#Designing an sgRNA
import re
import pandas as pd
from Bio.Seq import Seq

# Convert the genome from Cell 2 into a standard string
genome_str = str(genome_record.seq).upper()

# --- STEP 1: Select 20 nt target upstream of NGG PAM ---
pam_pattern = r"(?=([A-Z]{20})([A-Z]GG))"
matches = list(re.finditer(pam_pattern, genome_str))

# Create a dictionary to quickly count sequence occurrences for Step 2
all_spacers = [m.group(1) for m in matches]
spacer_counts = {}
for s in all_spacers:
    spacer_counts[s] = spacer_counts.get(s, 0) + 1

table_data = []

for index, match in enumerate(matches, start=1):
    spacer = match.group(1)
    pam = match.group(2)

    # --- STEP 2: Ensure target is unique in genome ---
    # Non-unique guides risk cutting off-target regions
    is_unique = "Yes" if spacer_counts[spacer] == 1 else "No"

    # --- STEP 3: Maintain ~40–60% GC content ---
    gc_count = spacer.count('G') + spacer.count('C')
    gc_percent = (gc_count / len(spacer)) * 100
    gc_status = "Optimal" if (40 <= gc_percent <= 60) else "Suboptimal"

    # --- STEP 4: Avoid long repeats (Homopolymers >= 4 bases) ---
    # Detects strings like AAAA, TTTT, GGGG, CCCC
    has_long_repeat = "Yes" if re.search(r"([A-Z])\1{3,}", spacer) else "No"

    # --- STEP 5: Write complementary gRNA sequence (5' to 3') ---
    # The transcribed RNA guide matches the target spacer sequence strand, replacing T with U
    grna_seq = spacer.replace("T", "U")

    # Determine overall viability based on your 5-step graphic rules
    is_viable = (is_unique == "Yes") and (gc_status == "Optimal") and (has_long_repeat == "No")

    # Calculate coordinate positioning
    pam_start = match.start() + 21
    pam_end = pam_start + 2

    table_data.append({
        "Target Site": index,
        "Step 1: 20nt Target": spacer,
        "PAM": pam,
        "Step 2: Unique?": is_unique,
        "Step 3: GC%": f"{gc_percent:.1f}% ({gc_status})",
        "Step 4: Long Repeat?": has_long_repeat,
        "Step 5: Final gRNA (5'->3')": grna_seq,
        "Position": f"{pam_start}–{pam_end}",
        "Viable Candidate": "PASS" if is_viable else "FAIL"
    })

# Convert to DataFrame for visualization
pipeline_df = pd.DataFrame(table_data)

# Filter the DataFrame to display only the absolute best candidates
best_candidates = pipeline_df[pipeline_df["Viable Candidate"] == "PASS"].head(10)

# Display the clean table with row indices hidden
best_candidates.style.hide(axis="index")


Target Site,Step 1: 20nt Target,PAM,Step 2: Unique?,Step 3: GC%,Step 4: Long Repeat?,Step 5: Final gRNA (5'->3'),Position,Viable Candidate
3,AAATCTGTGTGGCTGTCACT,CGG,Yes,45.0% (Optimal),No,AAAUCUGUGUGGCUGUCACU,100–102,PASS
5,GTAACTCGTCTATCTTCTGC,AGG,Yes,45.0% (Optimal),No,GUAACUCGUCUAUCUUCUGC,187–189,PASS
6,TATCTTCTGCAGGCTGCTTA,CGG,Yes,45.0% (Optimal),No,UAUCUUCUGCAGGCUGCUUA,197–199,PASS
7,AGCCGATCATCAGCACATCT,AGG,Yes,50.0% (Optimal),No,AGCCGAUCAUCAGCACAUCU,235–237,PASS
8,CAGCACATCTAGGTTTCGTC,CGG,Yes,50.0% (Optimal),No,CAGCACAUCUAGGUUUCGUC,245–247,PASS
9,AGCACATCTAGGTTTCGTCC,GGG,Yes,50.0% (Optimal),No,AGCACAUCUAGGUUUCGUCC,246–248,PASS
10,TTCGTCCGGGTGTGACCGAA,AGG,Yes,60.0% (Optimal),No,UUCGUCCGGGUGUGACCGAA,259–261,PASS
11,GGTGTGACCGAAAGGTAAGA,TGG,Yes,50.0% (Optimal),No,GGUGUGACCGAAAGGUAAGA,267–269,PASS
12,AAGATGGAGAGCCTTGTCCC,TGG,Yes,55.0% (Optimal),No,AAGAUGGAGAGCCUUGUCCC,283–285,PASS
15,GACGTGCTCGTACGTGGCTT,TGG,Yes,60.0% (Optimal),No,GACGUGCUCGUACGUGGCUU,358–360,PASS


In [ ]:
import re
import pandas as pd
import numpy as np
import plotly.express as px
from Bio.Seq import Seq
from Bio import Entrez, SeqIO

# ==========================================
# PART 1: EXTRACT CDS FEATURES & EARLY EXONS
# ==========================================
# Map all CDS ranges to find which genes our targets hit
cds_features = []
genome_len = len(genome_record.seq)

for feature in genome_record.features:
    if feature.type == "CDS":
        gene_name = feature.qualifiers.get("gene", [feature.qualifiers.get("product", ["Unknown"])[0]])[0]
        start = int(feature.location.start) + 1  # 1-based index translation
        end = int(feature.location.end)
        length = end - start + 1

        # Define the first 40% of the gene as the "Early Region" for ideal knockouts
        early_boundary = start + int(length * 0.40)

        cds_features.append({
            "gene": gene_name,
            "start": start,
            "end": end,
            "early_boundary": early_boundary
        })

def get_cds_annotation(pam_start):
    """Checks if a position hits a gene, and if it's within the early exon region."""
    for cds in cds_features:
        if cds["start"] <= pam_start <= cds["end"]:
            is_early = "Yes" if pam_start <= cds["early_boundary"] else "No"
            return cds["gene"], is_early
    return "Non-coding / Intergenic", "No"

# ==========================================
# PART 2: BASIC OFF-TARGET CHECKER
# ==========================================
# Download a secondary variant strain (e.g., SARS-CoV-2 Omicron BA.1 variant - OM570283.1)
print("Fetching secondary sequence variant for off-target validation...")
Entrez.email = "your_email@example.com"
off_handle = Entrez.efetch(db="nucleotide", id="OM570283.1", rettype="gb", retmode="text")
variant_record = SeqIO.read(off_handle, "genbank")
off_handle.close()
variant_str = str(variant_record.seq).upper()

def check_off_targets(spacer_seq):
    """
    Checks for full or partial off-target matching in a secondary genome.
    Returns: (Perfect Matches, Partial Mismatches with 1-3 base tolerance)
    """
    # 1. Perfect Match Query
    perfect_matches = variant_str.count(spacer_seq)

    # 2. Relaxed Lookahead Search allowing up to 3 nucleotide mismatches
    # Generates a sliding window search across the variant strand
    mismatch_count = 0
    for i in range(len(variant_str) - 20):
        window = variant_str[i:i+20]
        # Calculate Hamming distance between target spacer and current window
        diffs = sum(1 for a, b in zip(spacer_seq, window) if a != b)
        if 1 <= diffs <= 3:
            mismatch_count += 1

    return perfect_matches, mismatch_count

# ==========================================
# PART 3: ADVANCED PIPELINE RE-RUN
# ==========================================
genome_str = str(genome_record.seq).upper()
pam_pattern = r"(?=([A-Z]{20})([A-Z]GG))"
matches = list(re.finditer(pam_pattern, genome_str))

# Global spacer frequency profiling
all_spacers = [m.group(1) for m in matches]
spacer_counts = {s: all_spacers.count(s) for s in set(all_spacers)}

table_data = []
print("Processing genomic annotations and screening off-target profiles...")

# Analyze a sample slice or the full genome (subsetting to first 200 for fast processing)
for index, match in enumerate(matches[:200], start=1):
    spacer = match.group(1)
    pam = match.group(2)
    pam_start = match.start() + 21
    pam_end = pam_start + 2

    # Core biological evaluation rules
    is_unique = "Yes" if spacer_counts[spacer] == 1 else "No"
    gc_percent = ((spacer.count('G') + spacer.count('C')) / 20) * 100
    is_ideal_gc = 40 <= gc_percent <= 60
    has_long_repeat = "Yes" if re.search(r"([A-Z])\1{3,}", spacer) else "No"

    # Annotation & Off-Target extraction functions
    gene_hit, is_early_exon = get_cds_annotation(pam_start)
    perfect_offs, partial_offs = check_off_targets(spacer)

    # Comprehensive Pass filtering logic
    is_viable = (is_unique == "Yes") and is_ideal_gc and (has_long_repeat == "No") and (is_early_exon == "Yes") and (perfect_offs <= 1)

    table_data.append({
        "Target Site": index,
        "20nt Target": spacer,
        "PAM": pam,
        "GC Content": gc_percent,
        "Unique": is_unique,
        "Gene Hit": gene_hit,
        "Early Region": is_early_exon,
        "Perfect Off-Targets": perfect_offs,
        "Partial Off-Targets": partial_offs,
        "Position": pam_start,
        "Viability": "PASS" if is_viable else "FAIL"
    })

df_advanced = pd.DataFrame(table_data)

# Export the entire processed run directly to a native CSV template
df_advanced.to_csv("crispr_target_pipeline_results.csv", index=False)
print("\nSuccess: Clean pipeline table exported as 'crispr_target_pipeline_results.csv'!")

# Display passing targets
passing_subset = df_advanced[df_advanced["Viability"] == "PASS"]
print(f"Discovered {len(passing_subset)} optimal target sites matching all constraints.")


Fetching secondary sequence variant for off-target validation...
Processing genomic annotations and screening off-target profiles...

Success: Clean pipeline table exported as 'crispr_target_pipeline_results.csv'!
Discovered 96 optimal target sites matching all constraints.


In [ ]:
# Create interactive scatter visualization charting spatial coordinate distribution
fig = px.scatter(
    df_advanced,
    x="Position",
    y="GC Content",
    color="Viability",
    color_discrete_map={"PASS": "#2ca02c", "FAIL": "#d62728"},
    hover_data=["Target Site", "Gene Hit", "Early Region", "Perfect Off-Targets", "20nt Target"],
    title="CRISPR Target Site Distribution & GC Content Profiles Across the Genome",
    labels={"Position": "Genomic Position Coordinate (bp)", "GC Content": "GC Percentage (%)"}
)

# Design visual layout additions
fig.update_layout(
    template="plotly_dark",
    hoverlabel=dict(bgcolor="white", font_size=12, font_family="Monospace"),
    xaxis=dict(rangeslider=dict(visible=True)) # Interactive zoom timeline
)

# Display chart within workspace
fig.show()

# Render interactive HTML asset deployment copy
fig.write_html("interactive_crispr_plot.html")
print("Interactive standalone visualization downloaded directly as 'interactive_crispr_plot.html'")


Interactive standalone visualization downloaded directly as 'interactive_crispr_plot.html'


In [ ]:
from Bio import SeqIO
record = SeqIO.read("/content/sequence.gb", "genbank")
print(record)

ID: NC_056066.1
Name: NC_056066
Description: Ovis aries strain OAR_USU_Benz2616 breed Rambouillet chromosome 13, ARS-UI_Ramb_v3.0, whole genome shotgun sequence
Database cross-references: BioProject:PRJNA739192, BioSample:SAMN17575729, Assembly:GCF_016772045.2
Number of features: 13
/molecule_type=DNA
/topology=linear
/data_file_division=CON
/date=30-OCT-2023
/accessions=['NC_056066', 'REGION:', '46638372..46658965']
/sequence_version=1
/keywords=['WGS', 'RefSeq']
/source=Ovis aries (sheep)
/organism=Ovis aries
/taxonomy=['Eukaryota', 'Metazoa', 'Chordata', 'Craniata', 'Vertebrata', 'Euteleostomi', 'Mammalia', 'Eutheria', 'Laurasiatheria', 'Artiodactyla', 'Ruminantia', 'Pecora', 'Bovidae', 'Caprinae', 'Ovis']
/comment=REFSEQ INFORMATION: The reference sequence is identical to
CM028716.1.
Assembly name: ARS-UI_Ramb_v2.0
The genomic sequence for this RefSeq record is from the
whole-genome assembly released by the University of Idaho on
2021/02/03. The original whole-genome shotgun projec

In [ ]:
record.features

[SeqFeature(SimpleLocation(ExactPosition(0), ExactPosition(20594), strand=1), type='source', qualifiers=...),
 SeqFeature(SimpleLocation(ExactPosition(0), ExactPosition(20594), strand=1), type='gene', qualifiers=...),
 SeqFeature(CompoundLocation([SimpleLocation(ExactPosition(0), ExactPosition(52), strand=1), SimpleLocation(ExactPosition(2482), ExactPosition(2580), strand=1), SimpleLocation(ExactPosition(16554), ExactPosition(20578), strand=1)], 'join'), type='mRNA', qualifiers=...),
 SeqFeature(CompoundLocation([SimpleLocation(ExactPosition(1), ExactPosition(52), strand=1), SimpleLocation(ExactPosition(16554), ExactPosition(20594), strand=1)], 'join'), type='mRNA', qualifiers=...),
 SeqFeature(SimpleLocation(ExactPosition(16564), ExactPosition(17335), strand=1), type='CDS', qualifiers=...),
 SeqFeature(SimpleLocation(ExactPosition(16564), ExactPosition(17335), strand=1), type='CDS', qualifiers=...),
 SeqFeature(SimpleLocation(ExactPosition(16564), ExactPosition(16636), strand=1), type

In [ ]:
from Bio import SeqFeature

In [ ]:
start_pos = SeqFeature.AfterPosition(5)
end_pos  = SeqFeature.BetweenPosition(9,left=8,right=9)
my_location = SeqFeature.SimpleLocation(start_pos,end_pos)
my_location

SimpleLocation(AfterPosition(5), BetweenPosition(9, left=8, right=9))

In [ ]:
int(my_location.start)

5

In [ ]:
int(my_location.end)

9

In [ ]:
#Exact Location
exact_location  = SeqFeature.SimpleLocation(5,9)
print(exact_location)

[5:9]


In [ ]:
#Location Testing
from Bio import SeqIO


In [ ]:
my_snp = 4350
record = SeqIO.read("/content/sequence.gb", "genbank")
for feature in record.features:
  if my_snp in feature.location:
    print(feature.type,feature.qualifiers.get("db_xref"))

source ['taxon:9940']
gene ['GeneID:493887']
